In [2]:
REPO <- "https://raw.githubusercontent.com/andrecesar00/predicarga_andre_diego_kaykhy/main/"
source("https://raw.githubusercontent.com/fractalarea/pi3-motor-de-busca/main/motor/motor03.R")
download.file(paste0(REPO, "config.R"), "config.R")
source("config.R")
docs   <- readRDS(gzcon(url(paste0(REPO, "estrutura/banco-de-dados/docs.rds"))))
origem <- readRDS(gzcon(url(paste0(REPO, "estrutura/banco-de-dados/origem.rds"))))
ix     <- montar(docs, cfg)
estado()

<!-- estado-R:inicio -->
## Estado do R ao fim da sessão
*gerado por `anexar_estado()` em 2026-10-25 04:56 · R version 4.6.1 (2026-06-24) · motor03 v1 2026-10-01 · não edite à mão*

### Objetos

| objeto | tipo | tamanho | valor / amostra |
|---|---|---|---|
| `cfg$minimo` | numeric | 1 | 200 |
| `cfg$grupo` | character | 1 | <nome do grupo> |
| `docs` | character, nomeado | 64 | d1 São Vicente, oficialmente E..., d2 São Vicente marca o início ..., d3 Com um PIB de mais de 6 bil..., d4 São Vicente preserva divers..., d5 A Ilha de Gohayó foi descob..., d6 Nas primeiras décadas do sé..., ... |
| `ix$tokens` | list | 64 | elementos: d1, d2, d3, d4, d5, d6, d7, d8, ... |
| `ix$vocab` | character | 2136 | -, –, —, "berço, "bica, "biquinha"., ... |
| `ix$tf` | matrix | 2136 × 64 | linhas: -, –, —, ... \| colunas: d1, d2, d3, d4, ... |
| `ix$N` | integer | 1 | 64 |
| `ix$df` | numeric, nomeado | 2136 | - 2, – 2, — 2, "berço 1, "bica 1, "biquinha". 1, ... |
| `ix$idf_tfidf` | numeric, nomeado 

In [3]:
# Cria a função de limpeza.
limpar <- function(x, acentos) {
  # Se solicitado, transforma letras acentuadas: ã → a, ç → c.
  if (acentos == "translit")
    x <- iconv(x, from = "UTF-8", to = "ASCII//TRANSLIT")

  x <- tolower(x)                   # Deixa tudo em minúsculas.
  x <- gsub("[^a-z0-9 ]", " ", x)   # Troca caracteres fora da regra por espaços.
  x <- gsub("\\s+", " ", x)         # Junta espaços repetidos.
  trimws(x)                        # Remove espaços das pontas.
}

# Limpa o texto e separa as palavras.
tok <- function(x, acentos) {
  unlist(strsplit(limpar(x, acentos), " "))
}

# Compara as duas opções numa frase.
limpar("Município de São Vicente", "manter")
limpar("Município de São Vicente", "translit")

[1] "munic pio de s o vicente"

[1] "municipio de sao vicente"

In [4]:
source("config.R")  # Recarrega a configuração editada.
cfg$acentos        # Deve mostrar "translit".
cfg$grupo          # Deve mostrar "Pessoa".

[1] "translit"

[1] "Predicarga"

In [5]:
# Mostra o primeiro documento original.
docs[[1]]

# Mostra as palavras depois da limpeza escolhida.
tok(docs[[1]], cfg$acentos)

# Compara a quantidade de palavras antes e depois.
c(
  antes  = length(tokenizar(docs[[1]])),
  depois = length(tok(docs[[1]], cfg$acentos))
)

[1] "São Vicente, oficialmente Estância Balneária de São Vicente, é um município brasileiro do estado de São Paulo e primeira cidade do Brasil. Pertencente à Região Metropolitana da Baixada Santista, sua população, no censo de 2022, era de 329 911 habitantes, sendo a terceira cidade mais populosa do litoral paulista, atrás apenas de Santos e Praia Grande. Sua área territorial é de 148,151 km², o que lhe confere uma densidade demográfica de 2 226,86 habitantes por km²."

[1] "sao"           "vicente"       "oficialmente"  "estancia"     
 [5] "balnearia"     "de"            "sao"           "vicente"      
 [9] "e"             "um"            "municipio"     "brasileiro"   
[13] "do"            "estado"        "de"            "sao"          
[17] "paulo"         "e"             "primeira"      "cidade"       
[21] "do"            "brasil"        "pertencente"   "a"            
[25] "regiao"        "metropolitana" "da"            "baixada"      
[29] "santista"      "sua"           "populacao"     "no"           
[33] "censo"         "de"            "2022"          "era"          
[37] "de"            "329"           "911"           "habitantes"   
[41] "sendo"         "a"             "terceira"      "cidade"       
[45] "mais"          "populosa"      "do"            "litoral"      
[49] "paulista"      "atras"         "apenas"        "de"           
[53] "santos"        "e"             "praia"         "grande"       
[57] "sua"           "area"          "territorial"   "e"            
[61] "de"            "148"           "151"           "km2"          
[65] "o"             "que"           "lhe"           "confere"      
[69] "uma"           "densidade"     "demografica"   "de"           
[73] "2"             "226"           "86"            "habitantes"   
[77] "por"           "km2"

antes depois 
    76     78

In [6]:
# Confere se algum documento falhou na conversão dos acentos.
sum(is.na(iconv(docs, from = "UTF-8", to = "ASCII//TRANSLIT")))

# Reúne as palavras de todos os documentos já limpos.
todos <- unlist(lapply(docs, function(x) tok(x, cfg$acentos)))

# Mostra as 20 palavras mais frequentes.
sort(table(todos), decreasing = TRUE)[1:20]

[1] 0

todos
     de       e       a       o      em      da      do     que     sao     foi 
    420     204     196     141     113     105      95      81      65      57 
     no    para     com vicente      na      se      os      um   sousa  afonso 
     57      56      55      53      47      42      38      36      35      34 

In [12]:
# Recarrega as decisões salvas.
source("config.R")

# Confere a lista: "sao" não deve aparecer.
cfg$stopwords

ERROR: Error in source("config.R"): config.R:24:15: unexpected ','
23:   # --- Aula 01: o que e um documento ------------------------------------
24:   minimo = 200,
                  ^


In [13]:
# Substitui o config.R local por este conteúdo.
writeLines(
  c(
    'cfg <- list(',
    '  minimo = 200,',
    '  limpar = TRUE,',
    '  acentos = "translit",',
    '  stopwords = c("de", "e", "a", "o", "em", "da", "do",',
    '                "que", "foi", "no", "para", "com"),',
    '  grupo = "Predicarga"',
    ')'
  ),
  "config.R"
)

# Carrega o arquivo corrigido.
source("config.R")

# Confere as configurações.
cfg

$minimo
[1] 200

$limpar
[1] TRUE

$acentos
[1] "translit"

$stopwords
 [1] "de"   "e"    "a"    "o"    "em"   "da"   "do"   "que"  "foi"  "no"  
[11] "para" "com" 

$grupo
[1] "Predicarga"

In [14]:
# Limpa o texto e remove as palavras da lista do grupo.
sem_stop <- function(x, stopwords, acentos) {
  t <- tok(x, acentos)
  t[!t %in% stopwords]
}

# Aplica a preparação a cada documento.
tokens_g <- lapply(
  docs,
  function(x) sem_stop(x, cfg$stopwords, cfg$acentos)
)

# Compara a quantidade de termos distintos.
c(
  bruto = length(ix$vocab),
  limpo = length(unique(unlist(tokens_g)))
)

bruto limpo 
 2136  1831

In [15]:
# Começa com um índice vazio.
postings <- list()

# Percorre cada documento.
for (d in names(tokens_g)) {
  # Considera cada termo apenas uma vez por documento.
  for (termo in unique(tokens_g[[d]])) {
    # Acrescenta o documento à lista desse termo.
    postings[[termo]] <- c(postings[[termo]], d)
  }
}

# Confere quantos termos foram indexados.
length(postings)

# Mostra os cinco termos presentes em mais documentos.
sort(lengths(postings), decreasing = TRUE)[1:5]

[1] 1831

sao      na vicente      os     por 
     36      34      30      30      26

In [16]:
# Guarda o índice invertido dentro de ix.
ix$postings <- postings

# Cria a busca AND.
busca_AND <- function(consulta, ix, cfg) {
  # Prepara a consulta com as mesmas regras dos documentos.
  termos <- sem_stop(consulta, cfg$stopwords, cfg$acentos)

  # Se algum termo não existe no índice, não há resultado.
  if (!all(termos %in% names(ix$postings)))
    return(character(0))

  # Retorna os documentos presentes em todas as listas.
  Reduce(intersect, ix$postings[termos])
}

In [17]:
# Mostra em quais documentos cada termo aparece.
postings[["monumentos"]]
postings[["historicos"]]
postings[["centro"]]

# Busca documentos que tenham os três termos juntos.
r <- busca_AND("monumentos historicos centro", ix, cfg)
r

# Mostra o artigo de origem dos documentos encontrados.
origem[r]

NULL

[1] "d2" "d4"

[1] "d6"

character(0)

named character(0)

In [18]:
# Mostra cada lista com o nome do termo.
list(
  monumentos = postings[["monumentos"]],
  historicos = postings[["historicos"]],
  centro = postings[["centro"]]
)

# Confere o resultado da consulta completa.
busca_AND("monumentos historicos centro", ix, cfg)

$monumentos
NULL

$historicos
[1] "d2" "d4"

$centro
[1] "d6"

character(0)

In [20]:
# Mostra os documentos de cada termo.
list(
  fundacao = postings[["fundacao"]],
  vila = postings[["vila"]],
  martim = postings[["martim"]],
  afonso = postings[["afonso"]]
)

# Busca documentos que contenham os quatro termos.
r <- busca_AND("fundacao vila martim afonso", ix, cfg)

# Mostra os resultados e os artigos de origem.
r
origem[r]

$fundacao
[1] "d4"

$vila
 [1] "d2"  "d3"  "d4"  "d7"  "d8"  "d11" "d12" "d13" "d15" "d16" "d17" "d24"
[13] "d35" "d36" "d37" "d38" "d39" "d42" "d48"

$martim
 [1] "d2"  "d7"  "d8"  "d9"  "d17" "d24" "d25" "d27" "d28" "d29" "d30" "d31"
[13] "d32" "d33" "d36" "d38" "d43" "d45" "d46" "d49"

$afonso
 [1] "d2"  "d7"  "d8"  "d9"  "d17" "d24" "d25" "d27" "d28" "d29" "d30" "d31"
[13] "d32" "d33" "d36" "d38" "d43" "d45" "d46" "d49" "d50"

character(0)

named character(0)

In [21]:
# Busca documentos que contenham os dois termos.
r <- busca_AND("fundacao vila", ix, cfg)

# Mostra os documentos encontrados.
r

# Mostra de qual artigo vieram.
origem[r]

# Mostra o texto para conferir os termos no parágrafo.
docs[r]

[1] "d4"

d4 
"São Vicente (São Paulo)"

d4 
"São Vicente preserva diversos sítios e edifícios históricos construídos desde o século XVI, os quais testemunham o processo de ocupação e desenvolvimento do país. Entre eles, destaca-se a Casa de Taipa, a primeira casa do Brasil, construída entre 1516 e 1520; o Parque Cultural Vila São Vicente, onde há apresentações no Teatro José Anchieta, espaço para apresentações artísticas e oficinas culturais que homenageia o jesuíta José de Anchieta, pioneiro da dramaturgia brasileira; a Igreja Matriz de São Vicente Mártir, no local onde havia sido construída a primeira igreja matriz do Brasil; o Engenho dos Erasmos, que marca o início da cultura canavieira nacional, a qual se iniciou na Capitania de São Vicente, a Biquinha de Anchieta, a Ponte Pênsil e a Encenação da fundação da Vila de São Vicente, considerada pelo Guinness World Records o \"maior espetáculo teatral realizado em areia de praia do mundo\". Abriga o São Vicente Atlético Clube. Devido à sua importância histórica, política e cultural, além de suas belezas naturais e praias, cultura e culinária caiçara, é um importante destino turístico do Brasil, sendo procurada, em especial, nos setores do turismo histórico, cultural, arquitetônico, ecoturismo e o turismo de sol e praia. Anualmente, recebe mais de 2 milhões de turistas e é reconhecida como uma Estância Balneária."

In [22]:
# Confere uma busca com resultado e outra sem resultado.
list(
  com_resultado = busca_AND("fundacao vila", ix, cfg),
  sem_resultado = busca_AND("historicos centro", ix, cfg)
)

$com_resultado
[1] "d4"

$sem_resultado
character(0)

In [23]:
# Confere os arquivos presentes na sessão.
list.files()

[1] "00_FICHA_PROJETO.md"          "aula03_parteD_consolidado.md"
[3] "config.R"                     "sample_data"

In [24]:
# Acrescenta o estado da sessão à ficha.
anexar_estado("00_FICHA_PROJETO.md")

# Acrescenta o mesmo estado ao consolidado.
anexar_estado("aula03_parteD_consolidado.md")

Estado do R anexado a 00_FICHA_PROJETO.md (1917 linhas de objetos).

Estado do R anexado a aula03_parteD_consolidado.md (1917 linhas de objetos).

